<a href="https://colab.research.google.com/github/shah-zeb-naveed/large-language-models/blob/main/Shahzeb_Fine_tune_SmolLM_135M_with_GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune SmolLM-135M with GRPO

❤️ Created by [@maximelabonne](https://twitter.com/maximelabonne).

In [ ]:
!pip install -qqq datasets==3.2.0 transformers==4.47.1 trl==0.14.0 peft==0.14.0 accelerate==1.2.1 bitsandbytes==0.45.2 wandb==0.19.7 --progress-bar off
# !pip install -qqq flash-attn --no-build-isolation --progress-bar off
!pip install -qU wandb bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.5 MB/s eta 0:00:00


In [ ]:
import bitsandbytes
bitsandbytes.__version__

'0.50.2'

In [ ]:
import bitsandbytes.functional as F

print(F.__file__)
print(hasattr(F, "str2optimizer32bit"))

/usr/local/lib/python3.13/dist-packages/bitsandbytes/functional.py
False


In [ ]:
import torch
import wandb
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

# Log to Weights & Biases
wandb.login()

# Load dataset
dataset = load_dataset("mlabonne/smoltldr")
print(dataset)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: shahzebnaveed2 (shahzebnaveed2-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
})


In [ ]:
# Load model
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="sdpa",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load LoRA
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    # target_modules="all-linear",
)
model = get_peft_model(model, lora_config)
print(model.print_trainable_parameters())

trainable params: 230,400 || all params: 134,745,408 || trainable%: 0.1710
None


In [ ]:
import gc

# Reward function
def reward_len(completions, **kwargs):
    return [-abs(50 - len(completion)) for completion in completions]

# Training arguments
training_args = GRPOConfig(
    output_dir="GRPO",
    learning_rate=2e-5,
    per_device_train_batch_size=1, # Reduced batch size
    gradient_accumulation_steps=1,
    max_prompt_length=512,
    max_completion_length=64,
    num_generations=2, # Reduced number of generations
    optim="adamw_8bit",
    num_train_epochs=1,
    max_steps=100,
    # T4 = fp16, NOT bf16
    # fp16=True,
    bf16=True,
    report_to=["wandb"],
    remove_unused_columns=False,
    logging_steps=1,
    gradient_checkpointing=True, # Enabled gradient checkpointing
    gradient_checkpointing_kwargs={'use_reentrant': False}, # Required for some versions
)

# Trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_len],
    args=training_args,
    train_dataset=dataset["train"],
)

# Train model
wandb.init(project="SmolGRPO")
trainer.train()

# Clear GPU memory after training
del trainer
torch.cuda.empty_cache()
gc.collect()

train/completion_length,█▆█▇███▁████▅██▆████▅█████▅▆▅█▇▆▄█████▄█
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▇▇▇▇▇▇▇▇▇█
train/global_step,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
train/grad_norm,▄▅▄▃▄▁▃▆▃▂▅▂▃▃▃▃▄▄█▄▃▃▄▇▄▃▃▃▅▂▃▃▆▆▃▄▅▄▂█
train/kl,▆▇▂▅▅▄▅▃▆▄▃▆▁▅▅▇▄▆▄▃▅▆▅▅▅▃▃▄▅▅▃▄▃▆▁▃▇▄▂█
train/learning_rate,████▇▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
train/loss,█▁▁▁▁▁▁▁██▁▁▁▁▁▁▁▁█▁▁▁██▁█▁▁▁██▁▁▁▁▁▁▁▁▁
train/reward,▃▂▅▄▂▂▄▄▃▇▁▂▂▃▄▃▅▂▅▆▃▄▂▃▄▂▄█▃▃▃▄▄▁▄▄▂▂▄▇
train/reward_std,█▄▃▁▁▂▃▁█▁▁▄▃▃▁▁▃▂▃▂▃▂▂▄▃▅▁▂▁▄▁▃▂▁▂▂▅▁▂▂
train/rewards/reward_len,▄█▃▅▃▂▃▂▄▁▄▃▃▄▆▆▄▇▁▃▁▁▆▅▄▃▂▄▃▃▂▄▃▄▂▁▄▃▇▃
train/completion_length,64


Step,Training Loss
1,0.000100
2,0.000000
3,0.000100
4,0.000100
5,0.000000
6,0.000000
7,0.000000
8,0.000100
9,0.000000
10,0.000100


16236

In [ ]:
# Generate text
prompt = dataset["test"]["prompt"][0]
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output_ids = model.generate(
        **inputs, max_new_tokens=256, do_sample=True, temperature=0.5, min_p=0.1
    )
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)[
    len(prompt) :
]
print(f"TL;DR: {generated_text.strip()}")

TL;DR: I'm a girl who wants to let a guy know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and I'm a girl who wants to let him know that he's hurt and


In [ ]:
# Save model
merged_model = trainer.model.merge_and_unload()
merged_model.push_to_hub("SmolGRPO-135M")
trainer.tokenizer.push_to_hub("SmolGRPO-135M")

NameError: name 'trainer' is not defined